### 🧠 What is Query Decomposition?
Query decomposition is the process of taking a complex, multi-part question and breaking it into simpler, atomic sub-questions that can each be retrieved and answered individually.

#### ✅ Why Use Query Decomposition?

- Complex queries often involve multiple concepts

- LLMs or retrievers may miss parts of the original question

- It enables multi-hop reasoning (answering in steps)

- Allows parallelism (especially in multi-agent frameworks)

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence

In [3]:
#Step 1: Load documents and split the dataset into chunks
loader = TextLoader("langchain_crewai_dataset.txt")
raw_documents = loader.load()

# splitting the document into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 40)
chunks = splitter.split_documents(raw_documents)

In [7]:
# Step 2 
# Creating Vector store
embedding_model = HuggingFaceEmbeddings(model = "all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embedding_model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
# MMR retriever
retriever = vector_store.as_retriever(search_typre = "mmr" , search_kwargs = {"k":5})
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000176849B6FD0>, search_kwargs={'k': 5})

In [9]:
# LLM & prompt 
import os
from dotenv import load_dotenv
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

llm = init_chat_model("openai:gpt-5-nano-2025-08-07")
llm

ChatOpenAI(output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x00000176849B7390>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000176849B7750>, root_client=<openai.OpenAI object at 0x00000176849B7110>, root_async_client=<openai.AsyncOpenAI object at 0x00000176849B74D0>, model_name='gpt-5-nano-2025-08-07', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_proxy=None, stream_usage=True)

In [10]:
# Step 3 : Query Decomposition
decomposition_prompt = PromptTemplate.from_template("""
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions:
""")

decomposition_chain = decomposition_prompt | llm | StrOutputParser()
decomposition_chain

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='\nYou are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.\n\nQuestion: "{question}"\n\nSub-questions:\n')
| ChatOpenAI(output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x00000176849B7390>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000176849B7750>, root_client=<openai.OpenAI object at 0x00000176849B7110>, root_async_client=<openai.AsyncOpenAI object at 0x00000176849B74D0>, model_name='gpt-5-nano-2025-08-07', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_proxy=None, stream_usage=True)
| StrOutputParser()

In [11]:
query = "How does LangChain use memory and agents compared to CrewAI?"
decomposition_question=decomposition_chain.invoke({"question": query})

In [12]:
print(decomposition_question)

- Sub-question 1: What memory primitives and storage options does LangChain offer (e.g., ConversationBufferMemory, ConversationSummaryMemory, EntityMemory, VectorStore-backed memories), and how can memory be persisted across sessions for agents?

- Sub-question 2: What memory primitives and storage options does CrewAI offer, including persistence, retrieval, and how memory is updated and accessed by agents?

- Sub-question 3: How are LangChain agents designed to operate (planning/decision-making with tools, use of memory during reasoning), and how is memory integrated into their workflow?

- Sub-question 4: How are CrewAI agents designed to operate (architecture, tool integration, planning), how do they leverage memory within agent workflows, and what are the main differences or trade-offs compared with LangChain?


In [13]:
# Step 4 : QA-chain per sun-question
qa_prompt = PromptTemplate.from_template("""
 Use the Context below to answer the question.
                                       
 Context : {context}
                                         
 Question : {Input}
 """
)

qa_chain = create_stuff_documents_chain(llm=llm , prompt=qa_prompt)

In [14]:
# Step 5 : Full RAG pipeline logic
def full_query_decomposition_pipeline(user_query):
    # Decompose the query
    sub_qs_text = decomposition_chain.invoke({"question":user_query})
    sub_questions = [q.strip("-.123456789. ").strip() for q in sub_qs_text.split("\n") if q.strip()]

    results = []
    for sub_q in sub_questions:
        docs = retriever.invoke(sub_q)
        result = qa_chain.invoke({"input" : sub_q , "context" : docs})
        results.append(f" Q: {sub_q}\nA: {result}")
    
    return "\n\n".join(results)


In [16]:
# Step 6: Run
query = "How does LangChain use memory and agents compared to CrewAI?"
final_answer = full_query_decomposition_pipeline(query)
print("✅ Final Answer:\n")
print(final_answer)

KeyError: "Input to PromptTemplate is missing variables {'Input'}.  Expected: ['Input', 'context'] Received: ['input', 'context']\nNote: if you intended {Input} to be part of the string and not a variable, please escape it with double curly braces like: '{{Input}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "